# Optimization Campaign (HITL)

Interactive notebook for running prompt optimization campaigns with full human-in-the-loop control.

**Workflow:** Config → Replay → Diagnostics → Baseline Eval → Optimization Round → LLM Suggestions → Repeat

**Prerequisites:** TermNorm running at `http://127.0.0.1:8000`, Groq API key set in `.env`

## 1. Setup

In [ ]:
#@title Setup & imports
import sys
import os
import json
import uuid
import random
from pathlib import Path
from datetime import datetime, timezone

import httpx
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from api.models.backend import BackendConnection, Execution, ExecutionResultItem
from api.models.prompt_state import PromptState, OptimizationDefaults
from api.services.project_store import ProjectStore
from api.services.backend_client import BackendClient

TERMNORM_URL = "http://127.0.0.1:8000"
BACKEND_ID = "termnorm-local"
EXPERIMENT_ID = "1_production_historical"

store = ProjectStore(base_dir=PROJECT_ROOT / ".promptpotter" / "projects")
client = BackendClient(TERMNORM_URL)

# Register backend (idempotent)
if not store.get_backend(BACKEND_ID):
    store.register_backend(BackendConnection(
        id=BACKEND_ID, name="TermNorm Local",
        backend_type="termnorm", base_url=TERMNORM_URL,
    ))

# Load experiment data
exp_data = store.load_sync(BACKEND_ID, f"experiments/{EXPERIMENT_ID}.json")
if not exp_data:
    print("No synced experiment data. Run: await client.sync_experiments(store, BACKEND_ID)")
else:
    queries = client.extract_replay_queries(exp_data)
    terms = client.extract_session_terms(exp_data)
    print(f"Experiment: {exp_data.get('experiment', {}).get('name', EXPERIMENT_ID)}")
    print(f"Queries: {len(queries)}  |  Session terms: {len(terms)}")

# Campaign state (accumulated across rounds)
campaign_rounds = []   # list of {round, prompt_state, accuracy, results}
print("Ready.")

## 2. Campaign Config

Edit this cell and re-run to change settings for replay, pipeline parameters, optimization, and the evaluation LLM. After each round, the LLM suggestion cell will print a modified config you can copy back here.

In [ ]:
campaign_config = {
    "replay": {
        "skip_llm_ranking": False,
        "query_limit": 0,               # 0=all, N=first N for quick test
        "delay_between": 2.0,
    },
    "pipeline_params": {
        "max_sites": 7,                  # Web: pages fetched
        "num_results": 20,               # Web: search results count
        "content_char_limit": 800,       # Web: chars per page
        "raw_content_limit": 5000,       # LLM1: research text input
        "profiling_temperature": 0.3,    # LLM1: temperature
        "profiling_max_tokens": 1800,    # LLM1: output limit
        "ranking_temperature": 0,        # LLM2: temperature
        "ranking_max_tokens": 4000,      # LLM2: output limit
        "ranking_sample_size": 20,       # LLM2: candidates to rerank
        "max_token_candidates": 20,      # Token matching: kept
        "relevance_weight_core": 0.7,    # Scoring: core vs spec weight
    },
    "optimization": {
        "n_variants": 5,
        "creativity": 0.7,
        "improvement_threshold": 0.01,
        "max_rounds": 3,
    },
    "eval_llm": {
        "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        "temperature": 0,
        "max_tokens": 4000,
    },
}

print(json.dumps(campaign_config, indent=2))

## 3. Run Replay

Replay queries against TermNorm with the configured settings. Uses cache when pipeline_params haven't changed.

In [ ]:
#@title Replay pipeline
rc = campaign_config["replay"]
pp = campaign_config["pipeline_params"]

VARIANT_LABEL = "full-pipeline" if not rc["skip_llm_ranking"] else "no-llm2"
PIPELINE_NOTATION = "LLM1-TokenMatch-LLM2" if not rc["skip_llm_ranking"] else "LLM1-TokenMatch"

replay_queries_list = queries[:rc["query_limit"]] if rc["query_limit"] else queries
total = len(replay_queries_list)

# --- Check for existing execution with matching parameters ---
_cached = None
if not pp:  # only use cache when no param overrides
    for _ex in store.list_executions(BACKEND_ID):
        if (_ex["experiment_id"] == EXPERIMENT_ID and
            _ex["variant_label"] == VARIANT_LABEL and
            _ex["pipeline_notation"] == PIPELINE_NOTATION):
            _cached = store.load_execution(BACKEND_ID, _ex["execution_id"])
            if _cached:
                break

if _cached:
    execution = _cached
    replay_results = [r.model_dump() for r in _cached.results]
    _hits = sum(1 for r in replay_results if r.get("predicted") == r["ground_truth"])
    print(f"Using cached execution {execution.execution_id}")
    print(f"  Queries: {len(replay_results)}")
    print(f"  hit@1: {_hits}/{len(replay_results)} ({_hits/len(replay_results)*100:.1f}%)")
else:
    execution_id = uuid.uuid4().hex[:12]
    if pp:
        print(f"Pipeline overrides: {pp}")
    print(f"Replaying {total} queries against {TERMNORM_URL}...")

    _hits = 0
    _pbar = tqdm(total=total, desc="Replay", unit="query")

    async def on_result(result, index, total):
        global _hits
        store.append_result(BACKEND_ID, execution_id, result)
        hit = result.get("predicted", "") == result["ground_truth"]
        if hit:
            _hits += 1
        done = index + 1
        tag = "HIT " if hit else "MISS"
        tqdm.write(
            f"[{done}/{total}] {tag}  {result['query'][:50]:<50s} "
            f"| pred: {result.get('predicted', '?')[:35]:<35s} "
            f"| Running: {_hits}/{done} ({_hits/done*100:.1f}%)"
        )
        _pbar.update(1)

    replay_results = await client.replay_queries(
        queries=replay_queries_list,
        terms=terms,
        skip_llm_ranking=rc["skip_llm_ranking"],
        delay_between=rc["delay_between"],
        on_result=on_result,
        pipeline_params=pp,
    )
    _pbar.close()

    successful = sum(1 for r in replay_results if r["status"] == "success")
    errors = sum(1 for r in replay_results if r["status"] == "error")
    execution = Execution(
        execution_id=execution_id,
        backend_id=BACKEND_ID,
        experiment_id=EXPERIMENT_ID,
        variant_label=VARIANT_LABEL,
        pipeline_notation=PIPELINE_NOTATION,
        session_terms_count=len(terms),
        pipeline_params=pp,
        query_count=len(replay_results),
        successful_count=successful,
        error_count=errors,
        results=[ExecutionResultItem(**r) for r in replay_results],
    )
    store.finalize_execution(execution)
    replay_results = [r if isinstance(r, dict) else r.model_dump() for r in replay_results]

# Summary
total_r = len(replay_results)
hits = sum(1 for r in replay_results if r.get("predicted") == r["ground_truth"])
avg_lat = sum(r.get("latency_ms", 0) for r in replay_results) / total_r if total_r else 0
avg_conf = sum(r.get("confidence", 0) for r in replay_results) / total_r if total_r else 0

print(f"\nReplay Summary")
print(f"  hit@1:          {hits}/{total_r} ({hits/total_r*100:.1f}%)")
print(f"  Avg latency:    {avg_lat:,.0f} ms")
print(f"  Avg confidence: {avg_conf:.3f}")

## 4. Diagnostic — Candidate Coverage

For each query: is the ground truth in the token-matched candidates? At what rank? This determines whether reranker optimization is viable (ground truth must be in the candidate set for the reranker to promote it).

In [ ]:
#@title Candidate coverage analysis
coverage_rows = []
for r in replay_results:
    if r.get("status") != "success":
        continue
    pd_data = r.get("pipeline_data", {})
    candidates = pd_data.get("token_matched_candidates", [])
    gt = r["ground_truth"]

    # candidates can be [(term, score), ...] or [term, ...]
    candidate_names = []
    for c in candidates:
        if isinstance(c, (list, tuple)):
            candidate_names.append(c[0])
        else:
            candidate_names.append(str(c))

    gt_rank = None
    for i, name in enumerate(candidate_names):
        if name == gt:
            gt_rank = i + 1
            break

    coverage_rows.append({
        "query": r["query"][:50],
        "ground_truth": gt[:40],
        "in_candidates": gt_rank is not None,
        "gt_rank": gt_rank,
        "num_candidates": len(candidate_names),
    })

cov_df = pd.DataFrame(coverage_rows)
covered = cov_df["in_candidates"].sum()
total_cov = len(cov_df)
coverage_pct = covered / total_cov * 100 if total_cov else 0

print(f"CANDIDATE COVERAGE")
print(f"="*50)
print(f"  Ground truth in candidates: {covered}/{total_cov} ({coverage_pct:.1f}%)")
print(f"  Missing from candidates:    {total_cov - covered}/{total_cov}")
print()

# Rank distribution
found = cov_df[cov_df["in_candidates"]]
if not found.empty:
    print(f"Rank distribution (ground truth position in candidate list):")
    print(f"  Rank 1 (already top):  {(found['gt_rank'] == 1).sum()}")
    print(f"  Rank 2-5:              {((found['gt_rank'] >= 2) & (found['gt_rank'] <= 5)).sum()}")
    print(f"  Rank 6-10:             {((found['gt_rank'] >= 6) & (found['gt_rank'] <= 10)).sum()}")
    print(f"  Rank 11-20:            {((found['gt_rank'] >= 11) & (found['gt_rank'] <= 20)).sum()}")
    print(f"  Rank >20:              {(found['gt_rank'] > 20).sum()}")
    print(f"  Mean rank:             {found['gt_rank'].mean():.1f}")
    print(f"  Median rank:           {found['gt_rank'].median():.0f}")

# Decision gate
print()
if coverage_pct > 50:
    print(f"DECISION: Coverage {coverage_pct:.0f}% > 50% threshold -> Reranker optimization is VIABLE.")
    print(f"  The ground truth exists in the candidate set; a better reranker prompt can promote it.")
else:
    print(f"DECISION: Coverage {coverage_pct:.0f}% <= 50% threshold -> Reranker optimization has LIMITED value.")
    print(f"  The ground truth is missing from candidates too often. Consider improving token matching first.")

In [ ]:
#@title Sample entity profiles (qualitative check)
n_samples = 3
samples = [r for r in replay_results if r.get("pipeline_data", {}).get("entity_profile")][:n_samples]

for i, s in enumerate(samples):
    profile = s["pipeline_data"]["entity_profile"]
    print(f"--- Sample {i+1}: {s['query'][:60]} ---")
    print(f"  Core concept: {profile.get('core_concept', '?')}")
    print(f"  Profile keys: {list(profile.keys())}")
    print(f"  Ground truth: {s['ground_truth']}")
    candidates = s.get("pipeline_data", {}).get("token_matched_candidates", [])[:5]
    print(f"  Top 5 candidates: {[c[0] if isinstance(c, (list,tuple)) else c for c in candidates]}")
    print()

## 5. Load Baseline & Evaluate

Load the current `llm_ranking` prompt from the synced experiment, wrap it in a PromptState, and evaluate it locally using cached pipeline data.

In [ ]:
#@title Load baseline reranker prompt
dependencies = exp_data.get("dependencies", {})
prompts = dependencies.get("prompts", {})

reranker_prompt = None
for key, prompt_info in prompts.items():
    if "llm_ranking" in key:
        reranker_prompt = prompt_info
        break

if reranker_prompt is None:
    raise RuntimeError(
        "No llm_ranking prompt found in synced experiment data. "
        "Re-sync the experiment after TermNorm prompt registry is initialized."
    )

baseline = PromptState(
    instruction=reranker_prompt["template"],
    parameters={
        "family": reranker_prompt.get("family", "llm_ranking"),
        "version": reranker_prompt.get("version"),
        "template_variables": reranker_prompt.get("template_variables", []),
    },
    changes_description="Baseline reranker_v1 from TermNorm prompt registry",
)

print(f"Baseline prompt loaded: {baseline.id[:12]}")
print(f"  Family: {baseline.parameters['family']}")
print(f"  Version: {baseline.parameters['version']}")
print(f"  Template length: {len(baseline.instruction)} chars")

In [ ]:
#@title Load evaluation data from replay results
# Use replay results that have entity_profile in pipeline_data
eval_data = [
    r for r in replay_results
    if r.get("status") == "success" and r.get("pipeline_data", {}).get("entity_profile")
]

print(f"Evaluation data: {len(eval_data)}/{len(replay_results)} queries with entity_profile")
if not eval_data:
    print("WARNING: No queries have entity_profile in pipeline_data.")
    print("Re-run replay (Section 3) with skip_llm_ranking=False.")

In [ ]:
#@title Define local_reranker_eval()
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
EVAL_LLM = campaign_config["eval_llm"]


async def local_reranker_eval(prompt_template: str, query_data: dict) -> dict:
    """Evaluate a reranker prompt on a single query using cached pipeline data.

    Returns dict with: query, predicted, ground_truth, hit, confidence, error
    """
    pipeline = query_data["pipeline_data"]
    entity_profile = pipeline["entity_profile"]
    candidates = pipeline.get("token_matched_candidates", [])
    ground_truth = query_data["ground_truth"]
    query = query_data["query"]

    core_concept = entity_profile.get("core_concept", "")
    entity_profile_json = json.dumps(entity_profile, indent=2)

    available = list(candidates[:20])
    sample_size = min(len(available), 20)
    sampled = random.sample(available, sample_size) if available else []
    matches = "\n".join(
        f"- {term}" if isinstance(term, str) else f"- {term[0]}"
        for term in sampled
    )

    rendered = prompt_template.replace("{{core_concept}}", str(core_concept))
    rendered = rendered.replace("{{entity_profile_json}}", entity_profile_json)
    rendered = rendered.replace("{{matches}}", matches)

    full_prompt = f"""{rendered}

IMPORTANT: Return a valid JSON response matching this exact structure:
{{
  "profile_summary": "Brief 1-2 sentence summary of the profile",
  "core_concept_description": "What the core concept fundamentally is",
  "ranked_candidates": [
    {{
      "candidate": "exact candidate string",
      "core_concept_score": 0.0,
      "spec_score": 0.0,
      "evaluation_reasoning": "Brief explanation without quotes or backslashes",
      "key_match_factors": ["factor1", "factor2"],
      "spec_gaps": ["gap1", "gap2"]
    }}
  ]
}}

Ensure all strings are properly escaped and avoid complex punctuation in reasoning."""

    try:
        async with httpx.AsyncClient() as http:
            resp = await http.post(
                EVAL_LLM["provider_url"],
                headers={
                    "Authorization": f"Bearer {GROQ_API_KEY}",
                    "Content-Type": "application/json",
                },
                json={
                    "model": EVAL_LLM["model"],
                    "messages": [{"role": "user", "content": full_prompt}],
                    "temperature": EVAL_LLM["temperature"],
                    "max_tokens": EVAL_LLM["max_tokens"],
                    "response_format": {"type": "json_object"},
                },
                timeout=60.0,
            )
            resp.raise_for_status()
            llm_output = resp.json()["choices"][0]["message"]["content"]
            parsed = json.loads(llm_output)

        ranked = parsed.get("ranked_candidates", [])
        top = ranked[0] if ranked else {}
        predicted = top.get("candidate", "NO_RESULT")
        confidence = top.get("core_concept_score", 0)

        return {
            "query": query,
            "predicted": predicted,
            "ground_truth": ground_truth,
            "hit": predicted == ground_truth,
            "confidence": confidence,
            "error": None,
        }
    except Exception as e:
        return {
            "query": query,
            "predicted": "ERROR",
            "ground_truth": ground_truth,
            "hit": False,
            "confidence": 0,
            "error": str(e),
        }


print(f"local_reranker_eval() defined  |  Model: {EVAL_LLM['model']}  |  API key set: {bool(GROQ_API_KEY)}")

In [ ]:
#@title Evaluate baseline prompt
assert eval_data, "No evaluation data -- run the 'Load evaluation data' cell first"

baseline_results = []
_pbar = tqdm(total=len(eval_data), desc="Baseline eval", unit="query")

for qd in eval_data:
    result = await local_reranker_eval(baseline.render(), qd)
    baseline_results.append(result)

    tag = "HIT " if result["hit"] else "MISS"
    hits_so_far = sum(1 for r in baseline_results if r["hit"])
    done = len(baseline_results)
    tqdm.write(
        f"[{done}/{len(eval_data)}] {tag}  {result['query'][:50]:<50s} "
        f"| pred: {result['predicted'][:35]:<35s} "
        f"| Running: {hits_so_far}/{done} ({hits_so_far/done*100:.1f}%)"
    )
    _pbar.update(1)

_pbar.close()

baseline_hits = sum(1 for r in baseline_results if r["hit"])
baseline_errors = sum(1 for r in baseline_results if r["error"])
baseline_accuracy = baseline_hits / len(baseline_results) if baseline_results else 0

# Record as round 0
campaign_rounds = [{
    "round": 0,
    "label": "baseline",
    "prompt_state": baseline,
    "accuracy": baseline_accuracy,
    "hits": baseline_hits,
    "total": len(baseline_results),
    "results": baseline_results,
}]

print(f"\nBASELINE EVALUATION")
print(f"  hit@1: {baseline_hits}/{len(baseline_results)} ({baseline_accuracy:.1%})")
print(f"  Errors: {baseline_errors}")

# Show failure examples
failures = [r for r in baseline_results if not r["hit"] and not r["error"]]
if failures:
    print(f"\nFailure examples ({len(failures)} total):")
    for r in failures[:5]:
        print(f"  Q: {r['query'][:55]}  |  Pred: {r['predicted'][:35]}  |  GT: {r['ground_truth'][:35]}")

## 6. Run One Optimization Round

Analyze failures from the current best, generate N candidate prompts, evaluate all locally, select the round winner. Re-run this section for additional rounds.

In [ ]:
#@title Run optimization round
opt = campaign_config["optimization"]
round_num = len(campaign_rounds)
current_best = campaign_rounds[-1]
current_ps = current_best["prompt_state"]
current_acc = current_best["accuracy"]
current_results = current_best["results"]

print(f"=== ROUND {round_num} ===")
print(f"Current best: {current_best['label']} ({current_acc:.1%})")
print()

# --- Analyze failures ---
failures = [r for r in current_results if not r["hit"] and not r["error"]]
failure_examples = "\n".join(
    f"  Query: {r['query'][:60]}  |  Predicted: {r['predicted'][:40]}  |  GT: {r['ground_truth'][:40]}"
    for r in failures[:15]
)

# --- Generate candidates via meta-prompt ---
n_variants = opt["n_variants"]
rendered_prompt = current_ps.render()

meta_prompt = f"""You are a prompt engineering expert. Generate {n_variants} improved variants
of a candidate-ranking prompt used in a terminology normalization pipeline.

CURRENT PROMPT (round {round_num - 1} winner -- {current_acc:.1%} accuracy on {len(current_results)} queries):
---
{rendered_prompt}
---

FAILURE EXAMPLES (predicted != ground_truth):
{failure_examples}

The prompt uses template variables (double-brace syntax):
  {{{{core_concept}}}} -- core concept from entity profile
  {{{{entity_profile_json}}}} -- full JSON entity profile from web research
  {{{{matches}}}} -- newline-separated list of "- candidate_term" from token matching

For each variant:
1. Analyze WHY the current prompt fails on the examples above
2. Make targeted changes to improve ranking accuracy (get correct candidate to rank #1)
3. Keep the same template variables and JSON output format

Return a JSON object with key "variants" containing an array of objects:
  - "variant_name": short identifier
  - "changes_description": 1-2 sentence description of what changed and why
  - "prompt_text": full prompt template text"""

print(f"Generating {n_variants} candidate prompts...")

async with httpx.AsyncClient() as http:
    resp = await http.post(
        EVAL_LLM["provider_url"],
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": EVAL_LLM["model"],
            "messages": [{"role": "user", "content": meta_prompt}],
            "temperature": opt["creativity"],
            "max_tokens": 16000,
            "response_format": {"type": "json_object"},
        },
        timeout=120.0,
    )
    resp.raise_for_status()
    raw = resp.json()["choices"][0]["message"]["content"]
    generated = json.loads(raw)

if isinstance(generated, dict):
    variants_list = generated.get("variants", generated.get("prompts", []))
else:
    variants_list = generated

candidates = []
for v in variants_list[:n_variants]:
    ps = current_ps.derive(
        instruction=v["prompt_text"],
        changes_description=v.get("changes_description", v.get("variant_name", "")),
    )
    candidates.append(ps)
    print(f"  {v.get('variant_name', ps.id[:12])}: {v.get('changes_description', '')[:80]}")

# --- Evaluate all candidates ---
print(f"\nEvaluating {len(candidates)} candidates on {len(eval_data)} queries...")

all_candidate_results = {}
for idx, candidate in enumerate(candidates):
    label = candidate.changes_description or candidate.id[:12]
    candidate_results = []
    _pbar = tqdm(total=len(eval_data), desc=f"Candidate {idx+1}", unit="query")

    for qd in eval_data:
        result = await local_reranker_eval(candidate.render(), qd)
        candidate_results.append(result)
        _pbar.update(1)

    _pbar.close()
    all_candidate_results[candidate.id] = candidate_results

    c_hits = sum(1 for r in candidate_results if r["hit"])
    c_acc = c_hits / len(candidate_results) if candidate_results else 0
    print(f"  [{idx+1}/{len(candidates)}] {label[:40]}: {c_hits}/{len(candidate_results)} ({c_acc:.1%})")

# --- Select round winner ---
best_acc = current_acc
best_ps = current_ps
best_results = current_results
best_label = current_best["label"]

for candidate in candidates:
    c_results = all_candidate_results[candidate.id]
    c_acc = sum(1 for r in c_results if r["hit"]) / len(c_results) if c_results else 0
    if c_acc > best_acc:
        best_acc = c_acc
        best_ps = candidate
        best_results = c_results
        best_label = candidate.changes_description or candidate.id[:12]

# Record round
round_entry = {
    "round": round_num,
    "label": best_label,
    "prompt_state": best_ps,
    "accuracy": best_acc,
    "hits": sum(1 for r in best_results if r["hit"]),
    "total": len(best_results),
    "results": best_results,
    "candidates_evaluated": len(candidates),
}
campaign_rounds.append(round_entry)

# --- Round summary table ---
print(f"\n{'='*70}")
print(f"ROUND {round_num} SUMMARY")
print(f"{'='*70}")

rows = []
rows.append({"prompt": f"current_best ({current_best['label'][:30]})", "hit@1": f"{current_acc:.1%}", "delta": "-"})
for candidate in candidates:
    c_results = all_candidate_results[candidate.id]
    c_acc = sum(1 for r in c_results if r["hit"]) / len(c_results) if c_results else 0
    delta = c_acc - current_acc
    rows.append({
        "prompt": (candidate.changes_description or candidate.id[:12])[:40],
        "hit@1": f"{c_acc:.1%}",
        "delta": f"{delta:+.1%}",
    })
display(pd.DataFrame(rows))

improved = best_acc > current_acc + opt["improvement_threshold"]
if improved:
    print(f"\nWINNER: {best_label} ({best_acc:.1%}, +{best_acc - current_acc:.1%} over previous)")
    print(f"  PromptState: {best_ps.id[:12]}  (parent: {best_ps.parent_id[:12] if best_ps.parent_id else 'none'})")
else:
    print(f"\nNo improvement beyond threshold ({opt['improvement_threshold']:.1%}). Keeping current best.")

## 7. LLM Suggestion for Next Round (HITL)

After each round, the LLM analyzes failures and suggests:
1. Failure pattern analysis
2. Parameter change suggestions
3. Prompt phrase fragments to adopt
4. Suggested next `campaign_config`

**Review the suggestions, edit the config cell (Section 2), then re-run Sections 6-7.**

In [ ]:
#@title Generate LLM suggestions for next round
current_best = campaign_rounds[-1]
current_ps = current_best["prompt_state"]
current_results = current_best["results"]
current_acc = current_best["accuracy"]

# Build failure detail with pipeline context
failures = [r for r in current_results if not r["hit"] and not r["error"]]
failure_detail = []
for r in failures[:20]:
    pd_data = next((rd.get("pipeline_data", {}) for rd in replay_results if rd["query"] == r["query"]), {})
    candidates = pd_data.get("token_matched_candidates", [])
    candidate_names = [c[0] if isinstance(c, (list, tuple)) else str(c) for c in candidates[:10]]
    gt_in_candidates = r["ground_truth"] in candidate_names
    profile = pd_data.get("entity_profile", {})

    failure_detail.append(
        f"  Query: {r['query'][:60]}\n"
        f"    Predicted: {r['predicted'][:50]}\n"
        f"    Ground truth: {r['ground_truth'][:50]}\n"
        f"    GT in candidates: {gt_in_candidates}\n"
        f"    Top candidates: {candidate_names[:5]}\n"
        f"    Core concept: {profile.get('core_concept', '?')}\n"
    )

# Round history
history_lines = []
for rd in campaign_rounds:
    history_lines.append(f"  Round {rd['round']}: {rd['label'][:40]} -> {rd['accuracy']:.1%}")

suggestion_prompt = f"""You are an expert optimization advisor for a terminology normalization pipeline.
Analyze the current campaign state and provide actionable suggestions for the next round.

CAMPAIGN HISTORY:
{chr(10).join(history_lines)}

CURRENT BEST PROMPT ({current_acc:.1%} accuracy):
---
{current_ps.render()}
---

CURRENT CONFIG:
{json.dumps(campaign_config, indent=2)}

FAILURE DETAILS ({len(failures)} failures out of {len(current_results)} queries):
{chr(10).join(failure_detail)}

Provide your analysis as a JSON object with these keys:

1. "failure_patterns": array of objects, each with:
   - "category": failure type (e.g., "bad_profile", "candidate_absent", "reranker_misjudged", "ambiguous_query")
   - "count": estimated count
   - "description": explanation
   - "examples": array of query strings

2. "parameter_suggestions": array of objects, each with:
   - "parameter": the pipeline_params key to change
   - "current_value": current value
   - "suggested_value": new value
   - "rationale": why this change helps

3. "prompt_phrase_fragments": array of objects, each with:
   - "action": one of "add_to_instruction", "modify_thinking_style", "add_few_shot", "modify_answer_format", "modify_persona"
   - "text": the exact text snippet to add or use
   - "rationale": why this helps with the observed failures

4. "suggested_config": a complete campaign_config JSON object with your recommended changes applied

5. "summary": 2-3 sentence overview of what to try next"""

print("Generating suggestions...")

async with httpx.AsyncClient() as http:
    resp = await http.post(
        EVAL_LLM["provider_url"],
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": EVAL_LLM["model"],
            "messages": [{"role": "user", "content": suggestion_prompt}],
            "temperature": 0,
            "max_tokens": 8000,
            "response_format": {"type": "json_object"},
        },
        timeout=120.0,
    )
    resp.raise_for_status()
    suggestions = json.loads(resp.json()["choices"][0]["message"]["content"])

# --- Display suggestions ---
print(f"\n{'='*70}")
print(f"LLM SUGGESTIONS FOR ROUND {len(campaign_rounds)}")
print(f"{'='*70}")

print(f"\nSUMMARY: {suggestions.get('summary', '')}")

# Failure patterns
print(f"\n--- FAILURE PATTERNS ---")
for fp in suggestions.get("failure_patterns", []):
    print(f"  [{fp.get('category', '?')}] ~{fp.get('count', '?')} queries: {fp.get('description', '')}")
    for ex in fp.get("examples", [])[:2]:
        print(f"    e.g. {ex[:60]}")

# Parameter suggestions
print(f"\n--- PARAMETER CHANGE SUGGESTIONS ---")
for ps in suggestions.get("parameter_suggestions", []):
    print(f"  {ps.get('parameter', '?')}: {ps.get('current_value', '?')} -> {ps.get('suggested_value', '?')}")
    print(f"    Rationale: {ps.get('rationale', '')}")

# Prompt phrase fragments
print(f"\n--- PROMPT PHRASE FRAGMENTS ---")
for pf in suggestions.get("prompt_phrase_fragments", []):
    print(f"  [{pf.get('action', '?')}]")
    print(f"    Text: \"{pf.get('text', '')}\"")
    print(f"    Rationale: {pf.get('rationale', '')}")
    print()

# Suggested config
print(f"\n--- SUGGESTED CONFIG (copy to Section 2) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

## 8. Campaign Summary

Compare all rounds, track per-query flips, display the PromptState lineage chain, and save the winner.

In [ ]:
#@title Campaign comparison table
rows = []
for rd in campaign_rounds:
    rows.append({
        "round": rd["round"],
        "label": rd["label"][:40],
        "hit@1": rd["hits"],
        "total": rd["total"],
        "accuracy": f"{rd['accuracy']:.1%}",
        "prompt_id": rd["prompt_state"].id[:12],
    })

print(f"CAMPAIGN SUMMARY ({len(campaign_rounds)} rounds)")
print(f"{'='*70}")
display(pd.DataFrame(rows))

In [ ]:
#@title Per-query flip tracking (baseline vs final)
if len(campaign_rounds) >= 2:
    base_r = campaign_rounds[0]["results"]
    final_r = campaign_rounds[-1]["results"]

    flips = []
    for br, fr in zip(base_r, final_r):
        b_hit = br["hit"]
        f_hit = fr["hit"]
        if b_hit != f_hit:
            flips.append({
                "query": br["query"][:50],
                "flip": "MISS->HIT" if f_hit else "HIT->MISS",
                "base_pred": br["predicted"][:35],
                "final_pred": fr["predicted"][:35],
                "ground_truth": br["ground_truth"][:35],
            })

    gained = sum(1 for f in flips if f["flip"] == "MISS->HIT")
    lost = sum(1 for f in flips if f["flip"] == "HIT->MISS")

    print(f"FLIP TRACKING (baseline -> round {campaign_rounds[-1]['round']})")
    print(f"  Queries gained (MISS->HIT): {gained}")
    print(f"  Queries lost (HIT->MISS):   {lost}")
    print(f"  Net change:                 {gained - lost:+d}")
    print()
    if flips:
        display(pd.DataFrame(flips))
else:
    print("Need at least 2 rounds for flip tracking.")

In [ ]:
#@title PromptState lineage chain
print("LINEAGE CHAIN")
print("="*50)
for i, rd in enumerate(campaign_rounds):
    ps = rd["prompt_state"]
    parent = ps.parent_id[:12] if ps.parent_id else "root"
    arrow = "  " if i == 0 else "  -> "
    print(f"{arrow}[{ps.id[:12]}] Round {rd['round']}: {rd['label'][:40]} ({rd['accuracy']:.1%})")
    if ps.parent_id:
        print(f"       parent: {parent}  |  changes: {ps.changes_description or 'none'}")

In [ ]:
#@title Save winner
winner = campaign_rounds[-1]["prompt_state"]
winner_acc = campaign_rounds[-1]["accuracy"]

# Find the actual best across all rounds
for rd in campaign_rounds:
    if rd["accuracy"] > winner_acc:
        winner = rd["prompt_state"]
        winner_acc = rd["accuracy"]

save_data = {
    "winner": winner.model_dump(),
    "accuracy": winner_acc,
    "campaign_rounds": len(campaign_rounds),
    "baseline_accuracy": campaign_rounds[0]["accuracy"],
    "improvement": winner_acc - campaign_rounds[0]["accuracy"],
    "config": campaign_config,
    "saved_at": datetime.now(timezone.utc).isoformat(),
}

filename = f"optimization/campaign_winner_{winner.id[:12]}.json"
store.save_sync(BACKEND_ID, filename, save_data)

print(f"WINNER SAVED")
print(f"  PromptState: {winner.id[:12]}")
print(f"  Accuracy: {winner_acc:.1%} (baseline: {campaign_rounds[0]['accuracy']:.1%}, delta: {save_data['improvement']:+.1%})")
print(f"  File: .promptpotter/projects/{BACKEND_ID}/sync/{filename}")
print(f"  Rounds completed: {len(campaign_rounds) - 1}")